# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Dataset DOI: [10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s as defined in the Croissant schema.

We list all available record sets, their fields, and columns, referencing by `@id` for clarity and reproducibility.

In [ ]:
# List all record sets by @id
from pprint import pprint

print("Record Sets found in dataset:")
record_set_ids = []
for record_set in dataset.metadata.record_sets:
    print(f"  - @id: {record_set['@id']}, name: {record_set.get('name', '')}")
    record_set_ids.append(record_set['@id'])

# For each record set, list its fields and columns referenced by @id
for record_set in dataset.metadata.record_sets:
    print(f"\nRecord Set @id: {record_set['@id']}, name: {record_set.get('name', '')}")
    if 'fields' in record_set:
        print("  Fields:")
        for field in record_set['fields']:
            print(f"    - @id: {field['@id']}, name: {field.get('name', '')}")
            if 'columns' in field:
                print("      Columns:")
                for col in field['columns']:
                    print(f"        - @id: {col['@id']}, name: {col.get('name', '')}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Below we demonstrate how to extract all rows from each available record set by referencing their `@id`.

Replace the chosen `record_set_id` and field/column IDs with the desired ones from the overview above as needed.

In [ ]:
# List of record set @ids discovered previously
record_sets = record_set_ids.copy()  # e.g. ['record_set_1', 'record_set_2', ...]
dataframes = {}

for record_set_id in record_sets:
    try:
        print(f"Loading records from record set @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        if not df.empty:
            print(f"Columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print("(Record set is empty)")
    except Exception as e:
        print(f"Could not load record set {record_set_id} due to error: {e}")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

Replace `<record_set_id>`, `<numeric_field_id>`, and `<group_field_id>` below with the actual `@id` strings from the record set and fields overview above as applicable to your EDA.

In [ ]:
# Example: Pick a record set with numeric fields
example_record_set_id = record_sets[0] if record_sets else None
if example_record_set_id:
    df = dataframes[example_record_set_id]

    print(f"Columns in {example_record_set_id}: {df.columns.tolist()}")

    # Try to auto-detect a numeric field (float or int)
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

    # If not, prompt user to set this field or check schema
    if numeric_field:
        print(f"Analyzing numeric field (by @id or column): {numeric_field}")
        threshold = df[numeric_field].mean()  # Use mean as example threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.4f}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to auto-detect a categorical/group field
        group_field = None
        # Look for type object or low cardinality
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and df[col].nunique() < df.shape[0] / 10:
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric field was detected in this record set. Please update the code to select a target field.")
else:
    print("No record sets available for EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot a histogram of the first detected numeric field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in {example_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
In this notebook we:
- Loaded and explored the metadata from the Croissant-based dataset describing mass spectrometry results of extracellular vesicles from human mast cells
- Discovered available record sets and their fields via `@id`
- Loaded the tabular data into pandas DataFrames for flexible analysis
- Performed basic EDA and visualized the distribution of a key numeric attribute

This notebook can be extended by customizing field selection, adding domain-specific analysis, and utilizing group-by or join operations as appropriate for the biological and proteomics context. Consider using the Croissant field and column `@id`s for reproducible subset and integration of data.